# Small coherent cat and scaling benchmark

**Publication copy.** Source code is unchanged from the saved 18 August 2026 notebook; only saved Jupyter outputs were cleared. It performs $R=100$ independent reconstructions for each $n=500,1000,2000$ and is computationally expensive. See `../docs/REPRODUCIBILITY.md`. The original notebook contains an additional optional historical experimental cell which skips itself if `DataPhase28.dat` is absent.


# Final KQSE / GMM-3 benchmark: R=100, new-approx bandwidth, frozen floor/clip, sign fixes

This notebook is the long production run for the $a=1+0.5i$ benchmark.

**Frozen numerical choices:** reconstruction/reference grids, KDE/Silverman pilot,
`phi_abs_floor=0.03`, clipping `0.01 <= h <= 0.5`, GMM-3 pipeline, observables,
metrics, checkpointing, and all other numerical settings are unchanged.

**Only scientific change relative to the previous R=100 notebook:**
the KCFE bandwidth uses the newer approximation
$h \propto [|t|\sqrt{\mu^2+\nu^2}]^{-1}$ with no extra factor $1/\sqrt{2}$.

The previously validated sign fixes are retained unchanged.

The synthetic run automatically performs 100 independent repetitions for each
$n\in\{500,1000,2000\}$ and checkpoints after every completed repetition.


In [ ]:
import os
import pickle
from pathlib import Path
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm.auto import tqdm
from scipy.integrate import simpson
from scipy.stats import gaussian_kde
from scipy.special import eval_laguerre
from sklearn.mixture import GaussianMixture

plt.rcParams["figure.dpi"] = 120

# ------------------------------------------------------------------
# FINAL SETTINGS: a = 1 + 0.5 i
# ------------------------------------------------------------------
SEED = 0
np.random.seed(SEED)

SAMPLE_SIZES = [500, 1000, 2000]
N_REPEATS = 100
ARTICLE_N = 1000

R_EST = 8.0
N_EST = 41
N_PRECISION = 1200
GMM_COMPONENTS = 3
R_LIST = [0, 1, 2, 3]

# Frozen historical KCFE regularization.
PHI_ABS_FLOOR = 0.03
H_MIN = 0.01
H_MAX = 0.5

# ------------------------------------------------------------------
# Checkpoint/output directory
# ------------------------------------------------------------------
# Recommended in Colab: True. The checkpoint survives runtime resets.
USE_GOOGLE_DRIVE = True

if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        OUTDIR = Path("/content/drive/MyDrive/KQSE_a1_R100_newapprox_floor003_clip_signfix")
    except ImportError:
        print("google.colab is unavailable; falling back to local article_outputs.")
        OUTDIR = Path("article_outputs_R100_newapprox_floor003_clip_signfix")
else:
    OUTDIR = Path("article_outputs_R100_newapprox_floor003_clip_signfix")

OUTDIR.mkdir(parents=True, exist_ok=True)

print("Synthetic sample sizes:", SAMPLE_SIZES)
print("Independent repetitions per n:", N_REPEATS)
print("Frozen bandwidth: phi_abs_floor =", PHI_ABS_FLOOR,
      ", h clip =", (H_MIN, H_MAX))
print("Output directory:", OUTDIR.resolve())


## Core KQSE model and bandwidth

In [ ]:
def W_0(mu, nu, X):
    rho2 = mu**2 + nu**2
    return 1.0 / np.sqrt(np.pi * rho2) * np.exp(-X**2 / rho2)


def phi_0(t, mu, nu):
    return np.exp(-t**2 * (mu**2 + nu**2) / 4.0)


def s_jk(j, k, mu, nu, alpha):
    return np.sqrt(2.0) * (
        (nu - 1j * mu) * np.conjugate(alpha) * np.exp(-2 * np.pi * 1j / 3.0 * (k - 1))
        - (nu + 1j * mu) * alpha * np.exp( 2 * np.pi * 1j / 3.0 * (j - 1))
    ) / (mu**2 + nu**2)


def d_jk(j, k, mu, nu, alpha):
    rho2 = mu**2 + nu**2
    return (
        -np.abs(alpha)**2
        + (
            (nu + 1j * mu)**2 * alpha**2 * np.exp( 4 * np.pi * 1j / 3.0 * (j - 1))
            + (nu - 1j * mu)**2 * np.conjugate(alpha)**2 * np.exp(-4 * np.pi * 1j / 3.0 * (k - 1))
        ) / (2.0 * rho2)
    )


def Nc_2(alpha):
    denom = 0.0 + 0.0j
    for j in range(1, 4):
        for k in range(1, 4):
            denom += np.exp(
                -np.abs(alpha)**2
                + np.abs(alpha)**2 * np.exp(2 * np.pi * 1j * (k - j) / 3.0)
            )
    return 1.0 / denom


def normalized_cat_state_pdf(mu, nu, alpha, X):
    summ = 0.0 + 0.0j
    for j in range(1, 4):
        for k in range(1, 4):
            summ += np.exp(1j * X * s_jk(j, k, mu, nu, alpha)) * np.exp(d_jk(j, k, mu, nu, alpha))
    return np.real(W_0(mu, nu, X) * Nc_2(alpha) * summ)


def normalize_pdf_on_grid(pdf, xgrid):
    pdf = np.asarray(pdf, dtype=float)
    pdf = np.maximum(pdf, 0.0)
    Z = np.trapezoid(pdf, xgrid)
    if (not np.isfinite(Z)) or (Z <= 0):
        raise ValueError("Bad PDF normalization.")
    return pdf / Z


def samplings_with_noisy_pdf(pdf, X, n, kappa=1.0):
    pdf = normalize_pdf_on_grid(pdf, X)
    samples = np.random.choice(X, size=n, p=pdf / np.sum(pdf))
    if kappa == 1:
        return samples
    return kappa * samples + (1 - kappa) * np.random.normal(0.0, 1.0, n)


def find_suitable_L(mu, nu, alpha, L_trial=60.0, n_grid=4000, tol_abs=1e-8, tol_rel=1e-4, safety=1.15):
    """
    Stable integration window:
    use the last significant support point, not the first local dip.
    """
    X = np.linspace(-L_trial, L_trial, n_grid)
    pdf_trial = np.real(normalized_cat_state_pdf(mu, nu, alpha, X))
    pdf_trial = np.maximum(pdf_trial, 0.0)

    thr = max(tol_abs, tol_rel * np.max(pdf_trial))
    mask = pdf_trial > thr

    if not np.any(mask):
        return min(10.0, L_trial)

    x_sig = X[mask]
    L = max(abs(x_sig[0]), abs(x_sig[-1]))
    L = min(L_trial, safety * L)
    return max(L, 6.0)


def pdf_kernel_optimal_bandwidth(X_grid, X_samples):
    """
    Gaussian KDE with Silverman bandwidth, matching the working notebook.
    """
    kde = gaussian_kde(X_samples, bw_method="silverman")
    w_values = kde.evaluate(X_grid)
    return normalize_pdf_on_grid(w_values, X_grid), float(kde.factor)


def compute_perfect_phi_safe(mu, nu, alpha, t=1.0, eps=1e-12):
    rho2 = mu**2 + nu**2
    if rho2 < eps:
        return 1.0 + 0.0j

    summ = 0.0 + 0.0j
    for j in range(1, 4):
        for k in range(1, 4):
            summ += phi_0(t + s_jk(j, k, mu, nu, alpha), mu, nu) * np.exp(d_jk(j, k, mu, nu, alpha))
    return Nc_2(alpha) * summ


def compute_phi_at_t(w_values, X_precision, t=1.0):
    integrand = w_values * np.exp(1j * X_precision * t)
    return np.trapezoid(integrand, x=X_precision)


def compute_gaussian_phi_at_t_optimal(
    mu, nu, X_precision, X_samples, w_gaussian_values, t,
    bd_min=H_MIN, bd_max=H_MAX, phi_abs_floor=PHI_ABS_FLOOR
):
    """
    Newer approximate KCFE bandwidth, with the historical stabilization retained.

    Frozen choices:
      - KDE/Silverman pilot for |phi|;
      - phi_abs_floor = 0.03;
      - clipping h to [0.01, 0.5].

    Bandwidth approximation:
        h = 2 / (|t| sqrt(mu^2+nu^2))
            * sqrt((1-|phi|^2)/(n |phi|^2)),
    followed by the historical clipping.
    """
    rho2 = mu**2 + nu**2
    if rho2 < 1e-12 or abs(t) < 1e-12:
        return 1.0 + 0.0j, 0.0

    phi_t = compute_phi_at_t(w_gaussian_values, X_precision, t)
    abs_phi = max(float(np.abs(phi_t)), float(phi_abs_floor))

    numer = max(0.0, 1.0 - abs_phi**2)
    denom = len(X_samples) * abs_phi**2

    raw_bd = 2.0 / (abs(t) * np.sqrt(rho2)) * np.sqrt(numer / denom)
    optimal_bd = float(np.clip(raw_bd, bd_min, bd_max))

    phi_K = np.exp(-optimal_bd**2 * t**2 * rho2 / 4.0)
    phi_nk = np.mean(np.exp(1j * np.asarray(X_samples) * t))

    return phi_nk * phi_K, optimal_bd


def compute_t_fourier_transform(pdf_values, X_precision, t=1.0):
    return simpson(pdf_values * np.exp(1j * X_precision * t), x=X_precision)


def symmetrize_phi_grid(phi_grid):
    phi = np.asarray(phi_grid, dtype=complex)
    return 0.5 * (phi + np.conj(phi[::-1, ::-1]))

def reconstruct_pdf_gmm_mle(X_samples, X_precision, gmm_components=3, random_state=0):
    X_samples = np.asarray(X_samples, float).ravel()
    X_grid = np.asarray(X_precision, float).ravel()

    gmm = GaussianMixture(
        n_components=gmm_components,
        covariance_type="full",
        n_init=5,
        reg_covar=1e-6,
        random_state=random_state
    ).fit(X_samples[:, None])

    pdf_hat = np.exp(gmm.score_samples(X_grid[:, None]))
    Z = simpson(pdf_hat, x=X_grid)
    if np.isfinite(Z) and Z > 0:
        pdf_hat /= Z
    else:
        raise ValueError("Bad GMM-MLE normalization.")
    return pdf_hat


## Kernel transforms and exact reference

In [ ]:
def centered_endpoint_excluded_grid(R, N):
    """
    Centered endpoint-excluded uniform grid on a window of length 2R.

    The grid has spacing Delta = 2R/N, contains the origin when N is odd,
    is symmetric under x -> -x, and avoids duplicate endpoint contributions
    in the periodic trapezoidal interpretation.
    """
    delta = 2.0 * float(R) / int(N)
    return (np.arange(int(N)) - (int(N) - 1) / 2.0) * delta


def photon_kernel(alpha_complex, mu, nu, r):
    radial = np.exp(-(mu**2 + nu**2) / 4.0)

    # SIGN FIX ONLY:
    # this is the alpha -> -alpha reflection required for consistency with
    # the stated dequantizer convention checked in the validation notebook.
    phase = np.exp(
        - ((nu - 1j * mu) / np.sqrt(2.0)) * np.conjugate(alpha_complex)
        + ((nu + 1j * mu) / np.sqrt(2.0)) * alpha_complex
    )

    lag = eval_laguerre(r, (mu**2 + nu**2) / 2.0)
    return radial * phase * lag


def build_phi_grid_exact(mu_grid, nu_grid, alpha_state, eps=1e-12):
    phi = np.zeros((len(mu_grid), len(nu_grid)), dtype=complex)
    for i, mu in enumerate(mu_grid):
        for j, nu in enumerate(nu_grid):
            phi[i, j] = compute_perfect_phi_safe(mu, nu, alpha_state, t=1.0, eps=eps)
    return phi


def wigner_from_phi_grid(phi_grid, mu_grid, nu_grid, q_grid, p_grid):
    dmu = mu_grid[1] - mu_grid[0]
    dnu = nu_grid[1] - nu_grid[0]

    W = np.zeros((len(q_grid), len(p_grid)), dtype=complex)
    MU, NU = np.meshgrid(mu_grid, nu_grid, indexing="ij")

    for iq, q in enumerate(tqdm(q_grid, desc="Wigner rows")):
        for ip, p in enumerate(p_grid):
            phase = np.exp(-1j * (MU * q + NU * p))
            W[iq, ip] = (dmu * dnu / (2 * np.pi)**2) * np.sum(phi_grid * phase)

    return np.real_if_close(W)


def photon_tomogram_from_phi_grid(phi_grid, mu_grid, nu_grid, alpha_eval_grid, r):
    dmu = mu_grid[1] - mu_grid[0]
    dnu = nu_grid[1] - nu_grid[0]

    WR = np.zeros(len(alpha_eval_grid), dtype=complex)
    MU, NU = np.meshgrid(mu_grid, nu_grid, indexing="ij")

    for ia, alpha_eval in enumerate(tqdm(alpha_eval_grid, desc=f"Photon tomogram r={r}")):
        kernel = photon_kernel(alpha_eval, MU, NU, r)
        WR[ia] = (dmu * dnu / (2 * np.pi)) * np.sum(phi_grid * kernel)

    return np.real_if_close(WR)

In [ ]:
def estimate_phi_point_kqse_from_samples(
    mu, nu, X_precision, X_samples,
    kappa=1.0, correction=False, eps=1e-12
):
    rho2 = mu**2 + nu**2
    if rho2 < eps:
        return 1.0 + 0.0j, 0.0

    W_gaussian, _ = pdf_kernel_optimal_bandwidth(X_precision, X_samples)

    if correction:
        phi_est, optimal_bd = compute_gaussian_phi_at_t_optimal(
            mu, nu, X_precision, X_samples, W_gaussian, t=1.0 / kappa
        )
        phi_est *= np.exp(((1.0 - kappa) / kappa)**2 / 2.0)
    else:
        phi_est, optimal_bd = compute_gaussian_phi_at_t_optimal(
            mu, nu, X_precision, X_samples, W_gaussian, t=1.0
        )

    return phi_est, optimal_bd


def estimate_phi_point_gmm_mle_from_samples(
    mu, nu, X_precision, X_samples,
    gmm_components=2, kappa=1.0, correction=False,
    random_state=0, eps=1e-12
):
    rho2 = mu**2 + nu**2
    if rho2 < eps:
        return 1.0 + 0.0j

    pdf_hat = reconstruct_pdf_gmm_mle(
        X_samples, X_precision,
        gmm_components=gmm_components,
        random_state=random_state
    )

    if correction and kappa != 1.0:
        phi_est = compute_t_fourier_transform(pdf_hat, X_precision, t=1.0 / kappa)
        phi_est *= np.exp(((1.0 - kappa) / kappa)**2 / 2.0)
    else:
        phi_est = compute_t_fourier_transform(pdf_hat, X_precision, t=1.0)

    return phi_est


def build_phi_grids_shared_samples(
    mu_grid, nu_grid, alpha_state, n_samples,
    n_precision=2000, gmm_components=2,
    kappa=1.0, correction=False, random_state=0, verbose=True
):
    phi_kqse = np.zeros((len(mu_grid), len(nu_grid)), dtype=complex)
    phi_mle  = np.zeros((len(mu_grid), len(nu_grid)), dtype=complex)
    bw_kqse  = np.zeros((len(mu_grid), len(nu_grid)), dtype=float)

    iterator_i = range(len(mu_grid))
    if verbose:
        iterator_i = tqdm(iterator_i, desc="Estimating phi-grids (shared samples)")

    for i in iterator_i:
        mu = mu_grid[i]
        for j, nu in enumerate(nu_grid):
            rho2 = mu**2 + nu**2
            if rho2 < 1e-12:
                phi_kqse[i, j] = 1.0 + 0.0j
                phi_mle[i, j] = 1.0 + 0.0j
                bw_kqse[i, j] = 0.0
                continue

            L_limit = find_suitable_L(mu, nu, alpha_state)
            X_precision = np.linspace(-L_limit, L_limit, n_precision)

            pdf_true = normalized_cat_state_pdf(mu, nu, alpha_state, X_precision)
            pdf_true = normalize_pdf_on_grid(pdf_true, X_precision)

            # ОДНА общая выборка для обоих методов
            X_samples = samplings_with_noisy_pdf(pdf_true, X_precision, n_samples, kappa)

            phi_kqse[i, j], bw_kqse[i, j] = estimate_phi_point_kqse_from_samples(
                mu, nu,
                X_precision=X_precision,
                X_samples=X_samples,
                kappa=kappa,
                correction=correction
            )

            phi_mle[i, j] = estimate_phi_point_gmm_mle_from_samples(
                mu, nu,
                X_precision=X_precision,
                X_samples=X_samples,
                gmm_components=gmm_components,
                kappa=kappa,
                correction=correction,
                random_state=random_state
            )

    return phi_kqse, bw_kqse, phi_mle

In [ ]:
alpha_state = 1.0 + 0.5j

R_REF = 8.0
N_REF = 241
mu_ref = centered_endpoint_excluded_grid(R_REF, N_REF)
nu_ref = centered_endpoint_excluded_grid(R_REF, N_REF)

q_grid = np.linspace(-4.0, 4.0, 81)
p_grid = np.linspace(-4.0, 4.0, 81)
alpha_eval_real = np.linspace(-4.0, 4.0, 121)
alpha_eval_grid = alpha_eval_real.astype(complex)

phi_ref = build_phi_grid_exact(mu_ref, nu_ref, alpha_state)
W_true = wigner_from_phi_grid(phi_ref, mu_ref, nu_ref, q_grid, p_grid)

w_true_dict = {
    r: photon_tomogram_from_phi_grid(
        phi_ref, mu_ref, nu_ref, alpha_eval_grid, r=r
    )
    for r in R_LIST
}

mu_est = centered_endpoint_excluded_grid(R_EST, N_EST)
nu_est = centered_endpoint_excluded_grid(R_EST, N_EST)

print("Reference and reconstruction grids prepared.")


## Synthetic Monte Carlo — automatic checkpointed $R=100$

One `Run all` starts all 100 repetitions for each sample size. If the runtime stops,
rerunning the notebook resumes from the saved Google Drive checkpoint.


In [ ]:
def run_single_publication_reconstruction(n_samples, random_state=0):
    phi_kqse, bw_kqse, phi_mle = build_phi_grids_shared_samples(
        mu_est, nu_est,
        alpha_state=alpha_state,
        n_samples=n_samples,
        n_precision=N_PRECISION,
        gmm_components=GMM_COMPONENTS,
        kappa=1.0,
        correction=False,
        random_state=random_state,
        verbose=False
    )

    phi_kqse = symmetrize_phi_grid(phi_kqse)
    phi_mle = symmetrize_phi_grid(phi_mle)

    i0 = np.argmin(np.abs(mu_est))
    j0 = np.argmin(np.abs(nu_est))
    phi_kqse[i0, j0] = 1.0 + 0.0j
    phi_mle[i0, j0] = 1.0 + 0.0j

    out = {"KQSE": {}, "GMM-3 MLE": {}}

    out["KQSE"]["W"] = wigner_from_phi_grid(
        phi_kqse, mu_est, nu_est, q_grid, p_grid
    )
    out["GMM-3 MLE"]["W"] = wigner_from_phi_grid(
        phi_mle, mu_est, nu_est, q_grid, p_grid
    )

    for r in R_LIST:
        out["KQSE"][f"w{r}"] = photon_tomogram_from_phi_grid(
            phi_kqse, mu_est, nu_est, alpha_eval_grid, r=r
        )
        out["GMM-3 MLE"][f"w{r}"] = photon_tomogram_from_phi_grid(
            phi_mle, mu_est, nu_est, alpha_eval_grid, r=r
        )

    return out, bw_kqse


targets = {"W": W_true, **{f"w{r}": w_true_dict[r] for r in R_LIST}}
cell_volumes = {
    "W": (q_grid[1] - q_grid[0]) * (p_grid[1] - p_grid[0]),
    **{f"w{r}": alpha_eval_real[1] - alpha_eval_real[0] for r in R_LIST},
}
labels = {"W": "Wigner", **{f"w{r}": f"w_{r}" for r in R_LIST}}
METHODS = ["KQSE", "GMM-3 MLE"]
OBS_KEYS = ["W"] + [f"w{r}" for r in R_LIST]

PIPELINE_TAG = "a1_R100_newapprox_floor003_clip001_05_signfix_v1"

CONFIG = {
    "pipeline_tag": PIPELINE_TAG,
    "seed": int(SEED),
    "alpha_state": complex(alpha_state),
    "sample_sizes": tuple(SAMPLE_SIZES),
    "n_repeats": int(N_REPEATS),
    "R_EST": float(R_EST),
    "N_EST": int(N_EST),
    "N_PRECISION": int(N_PRECISION),
    "GMM_COMPONENTS": int(GMM_COMPONENTS),
    "R_LIST": tuple(R_LIST),
    "phi_abs_floor": float(PHI_ABS_FLOOR),
    "h_min": float(H_MIN),
    "h_max": float(H_MAX),
    "q_grid": tuple(np.asarray(q_grid, float)),
    "p_grid": tuple(np.asarray(p_grid, float)),
    "alpha_eval_real": tuple(np.asarray(alpha_eval_real, float)),
}


def _new_accumulator():
    return {
        n: {
            method: {
                obs: {
                    "count": 0,
                    "sum_ise": 0.0,
                    # Required for Lubov's metric:
                    # average squared error pointwise FIRST, maximum SECOND.
                    "sum_err2": np.zeros_like(
                        np.asarray(targets[obs]), dtype=float
                    ),
                }
                for obs in OBS_KEYS
            }
            for method in METHODS
        }
        for n in SAMPLE_SIZES
    }


def _new_state():
    return {
        "version": 1,
        "config": CONFIG,
        "completed_reps": {n: [] for n in SAMPLE_SIZES},
        "accumulator": _new_accumulator(),
        "bandwidth_rows": [],
    }


def _atomic_pickle_dump(obj, path):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "wb") as f:
        pickle.dump(obj, f, protocol=pickle.HIGHEST_PROTOCOL)
    os.replace(tmp, path)


CHECKPOINT_PATH = OUTDIR / "synthetic_R100_checkpoint.pkl"


def _load_or_create_state():
    if CHECKPOINT_PATH.exists():
        with open(CHECKPOINT_PATH, "rb") as f:
            state = pickle.load(f)

        if state.get("config") != CONFIG:
            raise RuntimeError(
                "Existing checkpoint belongs to a different pipeline/configuration. "
                "Move/delete that checkpoint or use the matching notebook."
            )

        print("Loaded checkpoint:", CHECKPOINT_PATH)
        for n in SAMPLE_SIZES:
            print(f"  n={n}: {len(state['completed_reps'][n])}/{N_REPEATS} done")
        return state

    state = _new_state()
    _atomic_pickle_dump(state, CHECKPOINT_PATH)
    print("Created checkpoint:", CHECKPOINT_PATH)
    return state


def _add_reconstruction_to_state(state, n, global_rep, rec, bw):
    for method in METHODS:
        for obs in OBS_KEYS:
            err2 = np.abs(
                np.asarray(rec[method][obs]) - np.asarray(targets[obs])
            )**2
            ise = float(cell_volumes[obs] * np.sum(err2))

            a = state["accumulator"][n][method][obs]
            a["count"] += 1
            a["sum_ise"] += ise
            a["sum_err2"] += np.asarray(err2, dtype=float)

    state["bandwidth_rows"].append({
        "n": int(n),
        "global_rep": int(global_rep),
        "bw_mean": float(np.mean(bw)),
        "bw_median": float(np.median(bw)),
        "bw_min": float(np.min(bw)),
        "bw_max": float(np.max(bw)),
    })
    state["completed_reps"][n].append(int(global_rep))
    state["completed_reps"][n] = sorted(set(state["completed_reps"][n]))


def _metrics_from_accumulator(a, cell_volume):
    count = int(a["count"])
    if count == 0:
        return np.nan, np.nan

    pointwise_mse = np.asarray(a["sum_err2"], dtype=float) / count

    # Same reporting convention as the previous production notebook.
    mise = float(a["sum_ise"] / count)

    # Theorem-aligned finite-grid empirical metric:
    # max AFTER pointwise averaging over repetitions.
    mse_inf = float(np.max(pointwise_mse))

    mise_from_map = float(cell_volume * np.sum(pointwise_mse))
    if not np.isclose(mise, mise_from_map, rtol=1e-11, atol=1e-13):
        raise RuntimeError("MISE accumulator consistency check failed.")

    return mise, mse_inf


def _summary_table(state):
    rows = []
    for obs in OBS_KEYS:
        for method in METHODS:
            row = {"Observable": labels[obs], "Method": method}
            for n in SAMPLE_SIZES:
                a = state["accumulator"][n][method][obs]
                mise, mse_inf = _metrics_from_accumulator(a, cell_volumes[obs])
                row[f"R_n{n}"] = int(a["count"])
                row[f"MISE_n{n}"] = mise
                row[f"MSEinf_n{n}"] = mse_inf
            rows.append(row)
    return pd.DataFrame(rows)


# ------------------------------------------------------------------
# AUTOMATIC R=100 RUN / RESUME
# ------------------------------------------------------------------
state = _load_or_create_state()

todo = sum(
    rep not in set(state["completed_reps"][n])
    for n in SAMPLE_SIZES
    for rep in range(N_REPEATS)
)
progress = tqdm(total=todo, desc="Synthetic R=100 total remaining")

for n in SAMPLE_SIZES:
    completed = set(state["completed_reps"][n])

    for global_rep in range(N_REPEATS):
        if global_rep in completed:
            continue

        run_seed = int(SEED + 1000 * n + global_rep)
        np.random.seed(run_seed)

        rec, bw = run_single_publication_reconstruction(
            n_samples=n,
            random_state=run_seed
        )

        _add_reconstruction_to_state(
            state, n=n, global_rep=global_rep, rec=rec, bw=bw
        )

        if n == ARTICLE_N and global_rep == 0:
            _atomic_pickle_dump(
                rec,
                OUTDIR / f"representative_n{ARTICLE_N}_rep000.pkl"
            )

        # Save after EVERY finished (n, repetition).
        _atomic_pickle_dump(state, CHECKPOINT_PATH)

        # Also refresh a lightweight current summary.
        current_table = _summary_table(state)
        current_table.to_csv(
            OUTDIR / "synthetic_errors_current.csv",
            index=False
        )

        progress.update(1)

progress.close()

synthetic_table = _summary_table(state)

status = pd.DataFrame({
    "n": SAMPLE_SIZES,
    "R_completed": [len(state["completed_reps"][n]) for n in SAMPLE_SIZES],
    "R_target": [N_REPEATS] * len(SAMPLE_SIZES),
})
display(status)

synthetic_table_fmt = synthetic_table.copy()
for col in synthetic_table_fmt.columns:
    if col.startswith("MISE_") or col.startswith("MSEinf_"):
        synthetic_table_fmt[col] = synthetic_table_fmt[col].map(
            lambda x: f"{x:.3e}" if np.isfinite(x) else ""
        )
display(synthetic_table_fmt)

synthetic_table.to_csv(
    OUTDIR / "synthetic_errors_R100.csv",
    index=False
)
pd.DataFrame(state["bandwidth_rows"]).to_csv(
    OUTDIR / "synthetic_bandwidth_R100.csv",
    index=False
)
with open(
    OUTDIR / "synthetic_errors_R100.tex",
    "w",
    encoding="utf-8"
) as f:
    f.write(synthetic_table_fmt.to_latex(index=False, escape=False))

if all(len(state["completed_reps"][n]) == N_REPEATS for n in SAMPLE_SIZES):
    print("FINAL SYNTHETIC RUN COMPLETE: R=100 for n=500,1000,2000.")
else:
    print("Run is partial. Re-run the notebook; it will resume from the checkpoint.")


In [ ]:
REP_PATH = OUTDIR / f"representative_n{ARTICLE_N}_rep000.pkl"

article_figure_reconstruction = None
if REP_PATH.exists():
    with open(REP_PATH, "rb") as f:
        article_figure_reconstruction = pickle.load(f)

if article_figure_reconstruction is None:
    print("Representative reconstruction not available yet. "
          "It will be saved automatically when n=1000, rep=0 finishes.")
else:
    print("Loaded representative reconstruction:", REP_PATH)


## Experimental kitten-state reconstruction — same grids, sign conventions corrected

The experiment is run once if `DataPhase28.dat` is already available locally or in MyDrive.


In [ ]:
def data_wrapper(theta_list, x_list, N_theta):
    grouped_data_dict = {i: [] for i in range(N_theta)}

    theta_mod = np.mod(theta_list, np.pi)
    theta_even_odd = np.floor(theta_list / np.pi).astype(int) % 2
    bin_width = np.pi / N_theta
    bin_indices = np.floor(theta_mod / bin_width).astype(int)

    for bi, x, eo in zip(bin_indices, x_list, theta_even_odd):
        grouped_data_dict[int(bi)].append(float(x) * ((-1) ** (eo+1)))

    grouped_theta_list = []
    for i in range(N_theta):
        grouped_theta_list.append((i + 0.5) * bin_width)

    return grouped_theta_list, grouped_data_dict


def phi_1(theta, alpha, r2, radius, t=1, p=0.6810463):
    mu = radius * np.cos(theta)
    nu = radius * np.sin(theta)
    tau = np.sqrt(1-r2)
    c0 = tau / (np.sqrt(tau**2 + alpha**2))
    c1 = alpha / (np.sqrt(tau**2 + alpha**2))
    beta = np.abs(radius)
    phi0 = np.exp(- t**2 * beta**2 / 4)

    # SIGN FIX ONLY: +i agrees with the stated kitten tomogram/Wigner model.
    return phi0 * (
        1
        - p * c1**2 * beta**2 * t**2 / 2
        + 1j * np.sqrt(2) * p * c0 * c1 * t * mu
    )


def canonical_theta_radius_sample_sign(mu, nu, eps=1e-12):
    """
    Represent (mu,nu) by theta in [0,pi), non-negative radius, and a sample sign.

    (mu,nu) = sample_sign * radius * (cos(theta), sin(theta)).

    This is mathematically equivalent to the signed-radius convention tested in
    the validation notebook, but keeps t=radius >= 0 so the frozen historical
    bandwidth formula itself is not changed.
    """
    radius = float(np.hypot(mu, nu))
    if radius < eps:
        return 0.0, 0.0, 1.0

    theta_full = float(np.arctan2(nu, mu))
    if theta_full < 0.0:
        theta_full += 2.0 * np.pi

    if theta_full >= np.pi:
        theta = theta_full - np.pi
        sample_sign = -1.0
    else:
        theta = theta_full
        sample_sign = +1.0

    return theta, radius, sample_sign


def compute_gaussian_phi_at_t_optimal_stable(
    mu, nu, X_precision, X_samples, w_gaussian_values, t,
    bd_min=H_MIN, bd_max=H_MAX, phi_abs_floor=PHI_ABS_FLOOR
):
    # Same frozen new-approx h + floor/clip as in the synthetic part.
    return compute_gaussian_phi_at_t_optimal(
        mu, nu, X_precision, X_samples, w_gaussian_values, t,
        bd_min=bd_min, bd_max=bd_max, phi_abs_floor=phi_abs_floor
    )


def experimental_phi_point_kqse(
    mu, nu, grouped_data_dict, N_theta, X_precision,
    kappa=1.0, correction=False
):
    theta, radius, sample_sign = canonical_theta_radius_sample_sign(mu, nu)
    if radius < 1e-12:
        return 1.0 + 0.0j, 0.0

    theta_bin = int(np.floor(theta / (np.pi / N_theta)))
    theta_bin = min(max(theta_bin, 0), N_theta - 1)

    # SIGN FIX ONLY: absorb the signed-radius convention into the samples,
    # while keeping t=radius positive and leaving h untouched.
    xb = sample_sign * np.asarray(grouped_data_dict[theta_bin], dtype=float)

    w_gaussian, _ = pdf_kernel_optimal_bandwidth(X_precision, xb)

    mu_unit = np.cos(theta)
    nu_unit = np.sin(theta)

    if correction and kappa != 1.0:
        phi_est, bd = compute_gaussian_phi_at_t_optimal_stable(
            mu_unit, nu_unit, X_precision, xb, w_gaussian, t=radius / kappa
        )
        phi_est *= np.exp(((1.0 - kappa) / kappa)**2 / 2.0)
    else:
        phi_est, bd = compute_gaussian_phi_at_t_optimal_stable(
            mu_unit, nu_unit, X_precision, xb, w_gaussian, t=radius
        )

    return phi_est, bd


def experimental_phi_point_gmm(
    mu, nu, grouped_data_dict, N_theta, X_precision,
    gmm_components=2, kappa=1.0, correction=False, random_state=0
):
    theta, radius, sample_sign = canonical_theta_radius_sample_sign(mu, nu)
    if radius < 1e-12:
        return 1.0 + 0.0j

    theta_bin = int(np.floor(theta / (np.pi / N_theta)))
    theta_bin = min(max(theta_bin, 0), N_theta - 1)
    xb = sample_sign * np.asarray(grouped_data_dict[theta_bin], dtype=float)

    pdf_hat = reconstruct_pdf_gmm_mle(
        xb, X_precision,
        gmm_components=gmm_components,
        random_state=random_state
    )

    t_eff = radius / kappa if (correction and kappa != 1.0) else radius
    phi_est = compute_t_fourier_transform(pdf_hat, X_precision, t=t_eff)

    if correction and kappa != 1.0:
        phi_est *= np.exp(((1.0 - kappa) / kappa)**2 / 2.0)

    return phi_est


def build_phi_grid_experimental(
    mu_grid, nu_grid, grouped_data_dict, N_theta, X_precision,
    method="kqse", gmm_components=2, kappa=1.0, correction=False,
    random_state=0, verbose=True
):
    phi_grid = np.zeros((len(mu_grid), len(nu_grid)), dtype=complex)
    bw_grid = np.zeros((len(mu_grid), len(nu_grid)), dtype=float)

    iterator_i = range(len(mu_grid))
    if verbose:
        iterator_i = tqdm(iterator_i, desc=f"Building experimental phi-grid ({method})")

    for i in iterator_i:
        mu = mu_grid[i]
        for j, nu in enumerate(nu_grid):
            if method == "kqse":
                phi_grid[i, j], bw_grid[i, j] = experimental_phi_point_kqse(
                    mu, nu, grouped_data_dict, N_theta, X_precision,
                    kappa=kappa, correction=correction
                )
            elif method == "gmm":
                phi_grid[i, j] = experimental_phi_point_gmm(
                    mu, nu, grouped_data_dict, N_theta, X_precision,
                    gmm_components=gmm_components,
                    kappa=kappa, correction=correction,
                    random_state=random_state
                )
            else:
                raise ValueError("method must be 'kqse' or 'gmm'")

    return phi_grid, bw_grid


def build_phi_grid_model(mu_grid, nu_grid, alpha_model=0.3, r2_model=0.925, p_model=0.6810463):
    phi_grid = np.zeros((len(mu_grid), len(nu_grid)), dtype=complex)
    for i, mu in enumerate(mu_grid):
        for j, nu in enumerate(nu_grid):
            radius = np.hypot(mu, nu)
            if radius < 1e-12:
                phi_grid[i, j] = 1.0 + 0.0j
            else:
                theta = np.arctan2(nu, mu)
                phi_grid[i, j] = phi_1(
                    theta, alpha_model, r2_model, radius, t=1, p=p_model
                )
    return phi_grid


def enforce_phi_origin(phi_grid, mu_grid, nu_grid):
    out = phi_grid.copy()
    i0 = np.argmin(np.abs(mu_grid))
    j0 = np.argmin(np.abs(nu_grid))
    out[i0, j0] = 1.0 + 0.0j
    return out


In [ ]:
# Experimental part is run once, after the synthetic calculation.
# To make "Run all and leave" possible, this cell never opens an upload prompt.
# Put DataPhase28.dat in /content/ before Run all, or at MyDrive/DataPhase28.dat.

DATA_CANDIDATES = [
    Path("DataPhase28.dat"),
    Path("/content/DataPhase28.dat"),
    Path("/content/drive/MyDrive/DataPhase28.dat"),
]

DATA_PATH = next((p for p in DATA_CANDIDATES if p.exists()), None)
EXPERIMENT_READY = DATA_PATH is not None

if not EXPERIMENT_READY:
    print("DataPhase28.dat was not found. Synthetic R=100 is unaffected.")
    print("Experimental reconstruction is skipped.")
    print("For an automatic experimental run, place DataPhase28.dat at "
          "/content/DataPhase28.dat or MyDrive/DataPhase28.dat before Run all.")
else:
    print("Experimental data:", DATA_PATH)
    theta_list, x_list = np.loadtxt(DATA_PATH, unpack=True)

    N_theta = 20
    grouped_theta_list, grouped_data_dict = data_wrapper(theta_list, x_list, N_theta)

    exp_alpha_model = 0.3
    exp_r2_model = 0.925
    exp_p_model = 0.6810463

    exp_R_phi = 6.5
    exp_N_phi = 61
    exp_mu_grid = centered_endpoint_excluded_grid(exp_R_phi, exp_N_phi)
    exp_nu_grid = centered_endpoint_excluded_grid(exp_R_phi, exp_N_phi)
    exp_X_precision = np.linspace(-3.0, 3.0, 3000)

    phi_exp_kqse, bw_exp = build_phi_grid_experimental(
        exp_mu_grid, exp_nu_grid,
        grouped_data_dict=grouped_data_dict,
        N_theta=N_theta,
        X_precision=exp_X_precision,
        method="kqse",
        kappa=1.0,
        correction=False,
        verbose=True
    )
    phi_exp_kqse = enforce_phi_origin(
        symmetrize_phi_grid(phi_exp_kqse), exp_mu_grid, exp_nu_grid
    )

    phi_exp_gmm, _ = build_phi_grid_experimental(
        exp_mu_grid, exp_nu_grid,
        grouped_data_dict=grouped_data_dict,
        N_theta=N_theta,
        X_precision=exp_X_precision,
        method="gmm",
        gmm_components=GMM_COMPONENTS,
        kappa=1.0,
        correction=False,
        random_state=SEED,
        verbose=True
    )
    phi_exp_gmm = enforce_phi_origin(
        symmetrize_phi_grid(phi_exp_gmm), exp_mu_grid, exp_nu_grid
    )

    phi_model = build_phi_grid_model(
        exp_mu_grid, exp_nu_grid,
        alpha_model=exp_alpha_model,
        r2_model=exp_r2_model,
        p_model=exp_p_model
    )
    phi_model = enforce_phi_origin(phi_model, exp_mu_grid, exp_nu_grid)

    exp_q_grid = np.linspace(-4.0, 4.0, 81)
    exp_p_grid = np.linspace(-4.0, 4.0, 81)
    exp_alpha_eval = np.linspace(-5.0, 5.0, 121)

    W_model_exp = wigner_from_phi_grid(
        phi_model, exp_mu_grid, exp_nu_grid, exp_q_grid, exp_p_grid
    )
    W_kqse_exp = wigner_from_phi_grid(
        phi_exp_kqse, exp_mu_grid, exp_nu_grid, exp_q_grid, exp_p_grid
    )
    W_gmm_exp = wigner_from_phi_grid(
        phi_exp_gmm, exp_mu_grid, exp_nu_grid, exp_q_grid, exp_p_grid
    )

    w_model_exp = {
        r: photon_tomogram_from_phi_grid(
            phi_model, exp_mu_grid, exp_nu_grid, exp_alpha_eval, r=r
        )
        for r in R_LIST
    }
    w_kqse_exp = {
        r: photon_tomogram_from_phi_grid(
            phi_exp_kqse, exp_mu_grid, exp_nu_grid, exp_alpha_eval, r=r
        )
        for r in R_LIST
    }
    w_gmm_exp = {
        r: photon_tomogram_from_phi_grid(
            phi_exp_gmm, exp_mu_grid, exp_nu_grid, exp_alpha_eval, r=r
        )
        for r in R_LIST
    }

    print("Experimental reconstruction finished.")


In [ ]:
def ise_1d(reconstruction, target, grid):
    dx = float(grid[1] - grid[0])
    return float(dx * np.sum(np.abs(np.asarray(reconstruction) - np.asarray(target))**2))


def ise_2d(reconstruction, target, x_grid, y_grid):
    dx = float(x_grid[1] - x_grid[0])
    dy = float(y_grid[1] - y_grid[0])
    return float(dx * dy * np.sum(
        np.abs(np.asarray(reconstruction) - np.asarray(target))**2
    ))


def e_inf_prl(reconstruction, target):
    """PRL-style single-reconstruction squared uniform error: max_z |error(z)|^2."""
    return float(np.max(
        np.abs(np.asarray(reconstruction) - np.asarray(target))**2
    ))



if EXPERIMENT_READY:
    # Wigner figure
    vmin = float(np.min(np.real(W_model_exp)))
    vmax = float(np.max(np.real(W_model_exp)))

    fig, axes = plt.subplots(1, 3, figsize=(14.8, 5.0), constrained_layout=True)
    panels = [
        (W_model_exp, "Theory Model"),
        (W_kqse_exp, "KQSE"),
        (W_gmm_exp, "GMM-3 MLE"),
    ]
    for ax, (Wimg, title) in zip(axes, panels):
        im = ax.imshow(
            np.real(Wimg).T,
            origin="lower",
            extent=[exp_q_grid[0], exp_q_grid[-1], exp_p_grid[0], exp_p_grid[-1]],
            aspect="equal",
            vmin=vmin,
            vmax=vmax,
        )
        ax.set_title(title, fontsize=18)
        ax.set_xlabel(r"$q$", fontsize=17)
        ax.set_ylabel(r"$p$", fontsize=17)
        ax.tick_params(axis="both", labelsize=14)

    cbar = fig.colorbar(im, ax=axes, shrink=0.92)
    cbar.set_label(r"$W(q,p)$", fontsize=16)
    cbar.ax.tick_params(labelsize=14)
    fig.savefig(OUTDIR / "fig_wigner_experimental.pdf", bbox_inches="tight")
    plt.show()

    # Photon-number figure
    fig, axes = plt.subplots(2, 2, figsize=(12.8, 9.0), constrained_layout=True)
    axes = axes.ravel()

    for ax, r in zip(axes, R_LIST):
        ax.plot(exp_alpha_eval, np.real(w_model_exp[r]), linewidth=2.6, label="Theory Model")
        ax.plot(exp_alpha_eval, np.real(w_kqse_exp[r]), "--", linewidth=2.6, label="KQSE")
        ax.plot(exp_alpha_eval, np.real(w_gmm_exp[r]), ":", linewidth=2.8, label="GMM-3 MLE")
        ax.set_xlabel(r"$\mathrm{Re}\,\alpha$", fontsize=17)
        ax.set_ylabel(rf"$w_{{{r}}}(\alpha)$", fontsize=17)
        ax.tick_params(axis="both", labelsize=14)
        ax.grid(alpha=0.3)

    handles, legend_labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, legend_labels, loc="upper center", ncol=3, frameon=True)
    fig.savefig(OUTDIR / "fig_photon_experimental_r0123.pdf", bbox_inches="tight")
    plt.show()

    # Experimental data are a single fixed dataset, so these are model discrepancies:
    #   ISE     = integral |Theta_hat - Theta_model|^2
    #   E_inf   = max      |Theta_hat - Theta_model|^2
    # There is NO Monte-Carlo expectation here, hence no MISE or MSE_inf.
    exp_rows = [
        {
            "Observable": "Wigner",
            "Method": "KQSE",
            "ISE": ise_2d(W_kqse_exp, W_model_exp, exp_q_grid, exp_p_grid),
            "E_inf": e_inf_prl(W_kqse_exp, W_model_exp),
        },
        {
            "Observable": "Wigner",
            "Method": "GMM-3 MLE",
            "ISE": ise_2d(W_gmm_exp, W_model_exp, exp_q_grid, exp_p_grid),
            "E_inf": e_inf_prl(W_gmm_exp, W_model_exp),
        },
    ]

    for r in R_LIST:
        exp_rows.extend([
            {
                "Observable": f"w_{r}",
                "Method": "KQSE",
                "ISE": ise_1d(w_kqse_exp[r], w_model_exp[r], exp_alpha_eval),
                "E_inf": e_inf_prl(w_kqse_exp[r], w_model_exp[r]),
            },
            {
                "Observable": f"w_{r}",
                "Method": "GMM-3 MLE",
                "ISE": ise_1d(w_gmm_exp[r], w_model_exp[r], exp_alpha_eval),
                "E_inf": e_inf_prl(w_gmm_exp[r], w_model_exp[r]),
            },
        ])

    experimental_table = pd.DataFrame(exp_rows)
    experimental_table_fmt = experimental_table.copy()
    for col in ["ISE", "E_inf"]:
        experimental_table_fmt[col] = experimental_table_fmt[col].map(lambda x: f"{x:.3e}")

    display(experimental_table_fmt)

    experimental_table.to_csv(
        OUTDIR / "experimental_model_discrepancy_ISE_Einf.csv",
        index=False
    )
    with open(
        OUTDIR / "experimental_model_discrepancy_ISE_Einf.tex",
        "w",
        encoding="utf-8"
    ) as f:
        f.write(experimental_table_fmt.to_latex(index=False, escape=False))

    print("Experimental metrics saved: ISE + E_inf.")

else:
    print('Experimental metrics skipped because DataPhase28.dat is unavailable.')
